# 2. Dataset Preparation

In [14]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import os
import warnings
warnings.filterwarnings('ignore')

RANDOM_SEED = 42
TEST_SIZE = 0.20
SUBSAMPLE_FRAC = 0.10

np.random.seed(RANDOM_SEED)
print(f"Random seed: {RANDOM_SEED}")
print(f"Train/Test split: {100*(1-TEST_SIZE):.0f}/{100*TEST_SIZE:.0f}")

Random seed: 42
Train/Test split: 80/20


## 2.1 Load Data

In [15]:
df = pd.read_csv('data/matrix.csv')
print(f"Loaded data: {df.shape[0]:,} rows, {df.shape[1]:,} columns")
df.head()

Loaded data: 37,700 rows, 4,008 columns


,id,name,feat_0,feat_1,feat_2,feat_3,feat_4,feat_5,feat_6,feat_7,...,feat_3996,feat_3997,feat_3998,feat_3999,feat_4000,feat_4001,feat_4002,feat_4003,feat_4004,ml_target
0,0,Eiryyy,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1,shawflying,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,2,JpMCarrilho,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
3,3,SuhwanCha,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
4,4,sunilangadi2,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1


In [16]:
id_col = 'id'
name_col = 'name'
target_col = 'ml_target'

feature_cols = [c for c in df.columns if c.startswith('feat_')]
print(f"Number of features: {len(feature_cols)}")

X = df[feature_cols]
y = df[target_col]
print(f"X shape: {X.shape}, y shape: {y.shape}")

Number of features: 4005
X shape: (37700, 4005), y shape: (37700,)


## 2.2 Missing Value Handling

In [17]:
missing_counts = X.isnull().sum()
total_missing = missing_counts.sum()

print(f"Total missing values: {total_missing}")

if total_missing > 0:
    X = X.fillna(X.median())
    print("Missing values filled with column median.")
else:
    print("No missing values found.")

Total missing values: 0
No missing values found.


## 2.3 Feature Engineering

In [18]:
print(f"Number of features: {len(feature_cols)}")
if len(feature_cols) < 20:
    print("Feature engineering would be applied (< 20 features)")
else:
    print("Skipping feature engineering (features >= 20)")

Number of features: 4005
Skipping feature engineering (features >= 20)


## 2.4 Train/Test Split (Stratified)

In [19]:
X_trainval, X_test, y_trainval, y_test, idx_trainval, idx_test = train_test_split(
    X, y, df['id'],
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_SEED
)

print(f"TRAINVAL size: {len(X_trainval):,} ({100*(1-TEST_SIZE):.0f}%)")
print(f"TEST size: {len(X_test):,} ({100*TEST_SIZE:.0f}%)")

print(f"\nClass distribution verification:")
print(f"Original - Class 0: {(y==0).mean():.2%}, Class 1: {(y==1).mean():.2%}")
print(f"TRAINVAL - Class 0: {(y_trainval==0).mean():.2%}, Class 1: {(y_trainval==1).mean():.2%}")
print(f"TEST     - Class 0: {(y_test==0).mean():.2%}, Class 1: {(y_test==1).mean():.2%}")

TRAINVAL size: 30,160 (80%)
TEST size: 7,540 (20%)

Class distribution verification:
Original - Class 0: 74.17%, Class 1: 25.83%
TRAINVAL - Class 0: 74.17%, Class 1: 25.83%
TEST     - Class 0: 74.16%, Class 1: 25.84%


## 2.5 Create Dataset Variants

In [20]:
trainval_raw = pd.DataFrame(X_trainval)
trainval_raw['id'] = idx_trainval.values
trainval_raw['ml_target'] = y_trainval.values

test_raw = pd.DataFrame(X_test)
test_raw['id'] = idx_test.values
test_raw['ml_target'] = y_test.values

print("Variant A (Raw) prepared.")
print(f"TRAINVAL: {trainval_raw.shape}, TEST: {test_raw.shape}")

Variant A (Raw) prepared.
TRAINVAL: (30160, 4007), TEST: (7540, 4007)


In [21]:
scaler = StandardScaler()

X_trainval_scaled = scaler.fit_transform(X_trainval)
X_test_scaled = scaler.transform(X_test)

trainval_scaled = pd.DataFrame(X_trainval_scaled, columns=feature_cols)
trainval_scaled['id'] = idx_trainval.values
trainval_scaled['ml_target'] = y_trainval.values

test_scaled = pd.DataFrame(X_test_scaled, columns=feature_cols)
test_scaled['id'] = idx_test.values
test_scaled['ml_target'] = y_test.values

print("Variant B (Standardized) prepared.")
print(f"TRAINVAL_scaled: {trainval_scaled.shape}, TEST_scaled: {test_scaled.shape}")

Variant B (Standardized) prepared.
TRAINVAL_scaled: (30160, 4007), TEST_scaled: (7540, 4007)


## 2.6 Save Datasets

In [22]:
os.makedirs('data', exist_ok=True)

trainval_raw.to_csv('data/TRAINVAL.csv', index=False)
test_raw.to_csv('data/TEST.csv', index=False)
print(f"Saved: data/TRAINVAL.csv ({len(trainval_raw):,} rows)")
print(f"Saved: data/TEST.csv ({len(test_raw):,} rows)")

trainval_scaled.to_csv('data/TRAINVAL_scaled.csv', index=False)
test_scaled.to_csv('data/TEST_scaled.csv', index=False)
print(f"Saved: data/TRAINVAL_scaled.csv ({len(trainval_scaled):,} rows)")
print(f"Saved: data/TEST_scaled.csv ({len(test_scaled):,} rows)")

Saved: data/TRAINVAL.csv (30,160 rows)
Saved: data/TEST.csv (7,540 rows)
Saved: data/TRAINVAL_scaled.csv (30,160 rows)
Saved: data/TEST_scaled.csv (7,540 rows)


## 2.7 Subsampling for Rapid Prototyping

In [23]:
trainval_sub = trainval_scaled.sample(frac=SUBSAMPLE_FRAC, random_state=RANDOM_SEED)
test_sub = test_scaled.sample(frac=SUBSAMPLE_FRAC, random_state=RANDOM_SEED)

trainval_sub.to_csv('data/TRAINVAL_subsample.csv', index=False)
test_sub.to_csv('data/TEST_subsample.csv', index=False)

print(f"Saved: data/TRAINVAL_subsample.csv ({len(trainval_sub):,} rows, {SUBSAMPLE_FRAC:.0%})")
print(f"Saved: data/TEST_subsample.csv ({len(test_sub):,} rows, {SUBSAMPLE_FRAC:.0%})")

Saved: data/TRAINVAL_subsample.csv (3,016 rows, 10%)
Saved: data/TEST_subsample.csv (754 rows, 10%)


## 2.8 Class Imbalance Check

In [24]:
minority_ratio = y_trainval.value_counts(normalize=True).min()

print(f"Minority class ratio: {minority_ratio:.2%}")

if minority_ratio < 0.15:
    print("\n⚠️ CLASS IMBALANCE DETECTED (< 15%)")
    print("Consider applying resampling techniques:")
    print("  - SMOTE")
    print("  - Random oversampling")
    print("  - Random undersampling")
else:
    print("No class imbalance (minority >= 15%)")

Minority class ratio: 25.83%
No class imbalance (minority >= 15%)


## 2.9 Save Scaler for API Use

In [25]:
import joblib

os.makedirs('models', exist_ok=True)
joblib.dump(scaler, 'models/scaler.pkl')
print("Saved: models/scaler.pkl")

Saved: models/scaler.pkl


## Summary

In [26]:
print(f"""
CONFIG:
• Random seed: {RANDOM_SEED}
• Train/Test split: {100*(1-TEST_SIZE):.0f}/{100*TEST_SIZE:.0f}
• Subsample fraction: {100*SUBSAMPLE_FRAC:.0f}%

MODELS:
• models/scaler.pkl - StandardScaler for API deployment

STEPS TAKEN:
1. No missing values found
2. Skipped feature selection (features > 20)
3. Z-score scaling applied
4. Stratified split maintained class proportions
""")


CONFIG:
• Random seed: 42
• Train/Test split: 80/20
• Subsample fraction: 10%

MODELS:
• models/scaler.pkl - StandardScaler for API deployment

STEPS TAKEN:
1. No missing values found
2. Skipped feature selection (features > 20)
3. Z-score scaling applied
4. Stratified split maintained class proportions

